In [ ]:
# --- Cell 1: Environment setup for Module 4 (RUN ONCE after a fresh runtime) ---
from google.colab import drive
drive.mount('/content/drive', force_remount=True)

# Sirf woh versions install karein jo aapas mein theek kaam karte hain
%pip -q install --upgrade --force-reinstall \
  "numpy==2.1.3" "pandas==2.2.2" "scipy==1.14.1" "scikit-learn==1.5.2" \
  "joblib==1.4.2" "pyyaml" "yfinance" "pandas-ta"

# AHEM: Yeh naye software ko load karne ke liye runtime ko khud-ba-khud restart karega
import os
print("Sahi libraries install ho gayi hain. Ab unhein load karne ke liye runtime restart kiya ja raha hai...")
os.kill(os.getpid(), 9)

Mounted at /content/drive
  Preparing metadata (setup.py) ... done
  Installing build dependencies ... done
  Getting requirements to build wheel ... done
  Preparing metadata (pyproject.toml) ... done
Requested yfinance from https://files.pythonhosted.org/packages/f3/23/0e28fa29eba03f33d74c58296f301064930340622be34b008ed02d4486de/yfinance-0.1.91-py2.py3-none-any.whl has invalid metadata: Expected matching RIGHT_PARENTHESIS for LEFT_PARENTHESIS, after version specifier
    appdirs (>=1.4.4cryptography>=3.3.2)
            ~~~~~~~~~~^
Please use pip<24.1 if you need to use this version.
Requested yfinance from https://files.pythonhosted.org/packages/f3/23/0e28fa29eba03f33d74c58296f301064930340622be34b008ed02d4486de/yfinance-0.1.91-py2.py3-none-any.whl has invalid metadata: Expected matching RIGHT_PARENTHESIS for LEFT_PARENTHESIS, after version specifier
    appdirs (>=1.4.4cryptography>=3.3.2)
            ~~~~~~~~~~^
Please use pip<24.1 if you need to use this version.
Requested yfinance

In [1]:
# --- Cell 1 (after restart): imports + basics ---
from google.colab import drive
drive.mount('/content/drive', force_remount=True)

ROOT = "/content/drive/MyDrive/ARVisionGold"

import os, yaml, json, pandas as pd, numpy as np
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import accuracy_score, precision_recall_fscore_support, confusion_matrix, classification_report
import joblib

# sanity: print versions  (should be numpy 2.1.3, pandas 2.2.2, sklearn 1.5.x, scipy 1.14.x)
import scipy, sklearn
print("numpy:", np.__version__, "| pandas:", pd.__version__, "| sklearn:", sklearn.__version__, "| scipy:", scipy.__version__)

with open(f"{ROOT}/configs/paths.yaml") as f:
    P = yaml.safe_load(f)
with open(f"{ROOT}/configs/model.yaml") as f:
    MCFG = yaml.safe_load(f)

CSV_PATH   = P["data"]["processed_csv"]
MODEL_PATH = P["artifacts"]["model_rf"]
METRICS_JSON = f"{os.path.dirname(MODEL_PATH)}/metrics.json"

print("CSV:", CSV_PATH)
print("Model path:", MODEL_PATH)


Mounted at /content/drive
numpy: 2.2.6 | pandas: 2.3.3 | sklearn: 1.6.1 | scipy: 1.16.2
CSV: /content/drive/MyDrive/ARVisionGold/data/processed/gold_data.csv
Model path: /content/drive/MyDrive/ARVisionGold/artifacts/models/gold_price_predictor.joblib


In [2]:
df = pd.read_csv(CSV_PATH, parse_dates=["Date"])
df = df.sort_values("Date").reset_index(drop=True)

feature_cols = MCFG["features"]["include"]
target_col   = MCFG["target"]["name"]  # "NextCloseUp"

# Guard: ensure all needed features present
missing = [c for c in feature_cols if c not in df.columns]
if missing:
    raise ValueError(f"Missing feature columns in CSV: {missing}")

# time-series split (no shuffle)
test_size = MCFG["split"]["test_size"]
n_test = max(1, int(len(df) * test_size))

X = df[feature_cols].copy()
y = df[target_col].astype(int).copy()

X_train, y_train = X.iloc[:-n_test], y.iloc[:-n_test]
X_test,  y_test  = X.iloc[-n_test:], y.iloc[-n_test:]

len(df), X_train.shape, X_test.shape


(2446, (1957, 13), (489, 13))

In [3]:
rf_params = (MCFG.get("model", {}).get("rf_params", {})) or {}
rf = RandomForestClassifier(random_state=MCFG.get("seed", 42), **rf_params)
rf.fit(X_train, y_train)

# predictions
y_pred = rf.predict(X_test)
proba = None
if hasattr(rf, "predict_proba"):
    proba = rf.predict_proba(X_test)

acc = accuracy_score(y_test, y_pred)
prec, rec, f1, _ = precision_recall_fscore_support(y_test, y_pred, average="binary", zero_division=0, pos_label=1)

print(f"Accuracy: {acc:.4f} | Precision(UP): {prec:.4f} | Recall(UP): {rec:.4f} | F1(UP): {f1:.4f}")
print("\nConfusion matrix:\n", confusion_matrix(y_test, y_pred))
print("\nClassification report:\n", classification_report(y_test, y_pred, zero_division=0))


Accuracy: 0.4254 | Precision(UP): 0.7500 | Recall(UP): 0.0106 | F1(UP): 0.0209

Confusion matrix:
 [[205   1]
 [280   3]]

Classification report:
               precision    recall  f1-score   support

           0       0.42      1.00      0.59       206
           1       0.75      0.01      0.02       283

    accuracy                           0.43       489
   macro avg       0.59      0.50      0.31       489
weighted avg       0.61      0.43      0.26       489



In [4]:
os.makedirs(os.path.dirname(MODEL_PATH), exist_ok=True)
joblib.dump(rf, MODEL_PATH)

metrics = {
    "accuracy": float(acc),
    "precision_up": float(prec),
    "recall_up": float(rec),
    "f1_up": float(f1),
    "n_train": int(len(X_train)),
    "n_test": int(len(X_test)),
    "features": feature_cols,
}
with open(METRICS_JSON, "w") as f:
    json.dump(metrics, f, indent=2)

print("Model saved:", MODEL_PATH)
print("Metrics saved:", METRICS_JSON)


Model saved: /content/drive/MyDrive/ARVisionGold/artifacts/models/gold_price_predictor.joblib
Metrics saved: /content/drive/MyDrive/ARVisionGold/artifacts/models/metrics.json
